In [65]:
import os
import pandas as pd
import requests
import spotipy
from IPython.display import display
from spotipy.oauth2 import SpotifyClientCredentials
from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv(usecwd=True)
loaded = load_dotenv(dotenv_path=dotenv_path)
print(f"dotenv_loaded={loaded}, dotenv_path={dotenv_path}")

dotenv_loaded=True, dotenv_path=/Users/Marcy_Student/Desktop/Spotify_Project/.env


In [66]:
# setting up credentials
spotify_client_id = os.getenv("SPOTIFY_CLIENT_ID")
spotify_client_secret = os.getenv("SPOTIFY_CLIENT_SECRET") 
rapidapi_key = os.getenv("RAPIDAPI_KEY")
rapidapi_host = "track-analysis.p.rapidapi.com"

auth_manager = SpotifyClientCredentials(
    client_id=spotify_client_id,
    client_secret=spotify_client_secret,
)
sp = spotipy.Spotify(auth_manager=auth_manager)

headers = {
    "x-rapidapi-key": rapidapi_key,
    "x-rapidapi-host": rapidapi_host,
    "Content-Type": "application/json",
}

print("Spotify + RapidAPI configuration loaded")

Spotify + RapidAPI configuration loaded


In [67]:
# testing spotipy with my fav artist NF
from IPython.display import display
query = "artist:NF"
album_results = sp.search(q=query, type="album", limit=10)
track_results = sp.search(q=query, type="track", limit=10)
albums = [
    {
        "album": item["name"],
        "album_type": item["album_type"],
        "release_date": item["release_date"],
        "total_tracks": item["total_tracks"],
    }
    for item in album_results["albums"]["items"]
]
tracks = [
    {

        "track": item["name"],
        "album": item["album"]["name"],
        "artists": ", ".join(artist["name"] for artist in item["artists"]),
        "preview_url": item.get("preview_url"),
    }
    for item in track_results["tracks"]["items"]
]
print("NF albums")
display(pd.DataFrame(albums))
print("NF tracks")
display(pd.DataFrame(tracks))


NF albums


,album,album_type,release_date,total_tracks
0,HOPE,single,2023-02-16,1
1,WHO I WAS,single,2025-11-13,1
2,The Search,single,2019-05-30,1
3,FEAR,single,2025-11-14,6
4,HOPE,album,2023-04-07,13
5,CLOUDS (THE MIXTAPE),album,2021-03-26,11
6,Therapy Session,album,2016-04-22,14
7,Mansion,album,2015-03-31,12
8,Perception,album,2017-10-06,16
9,The Search,album,2019-07-26,20


NF tracks


,track,album,artists,preview_url
0,Let You Down,Perception,NF,None
1,HAPPY,HOPE,NF,None
2,FEAR,FEAR,NF,None
3,Lie,Perception,NF,None
4,The Search,The Search,NF,None


In [68]:
# creating a function to get the track informations of tracks_id
# My track Id will be store in a list
# This is a simple test for one track id
# the workflow will take export batch of track ids from spotify playlists endpoint\
track_ids = [
    "7s25THrKz86DM225dOYwnr"  # sample ID from SoundNet docs
]

def get_track_info(track_id):
    track = sp.track(track_id)

    return{
        "track_id": track_id,
        "track_name": track["name"],
        "artist": ",".join(a["name"] for a in track["artists"]),
        "album":track["album"]["name"],
    }
tracks_df = pd.DataFrame([get_track_info(track_id) for track_id in track_ids])
display(tracks_df)

,track_id,track_name,artist,album
0,7s25THrKz86DM225dOYwnr,Respect,Aretha Franklin,I Never Loved a Man the Way I Love You


In [71]:
feature_fields = [
    "key",
    "mode",
    "tempo",
    "camelot",
    "duration",
    "popularity",
    "energy",
    "danceability",
    "happiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "loudness",
]

results = []

for track_id in track_ids:
    try:
        resp = requests.get(
            f"https://track-analysis.p.rapidapi.com/pktx/spotify/{track_id}",
            headers=headers,
            timeout=20,
        )
        payload = resp.json()

        row = {
            "track_id": track_id,
            "status_code": resp.status_code,
            "error_message": payload.get("message") if resp.status_code != 200 else None,
        }

        if resp.status_code == 200:
            row.update({feature: payload.get(feature) for feature in feature_fields})
        else:
            row.update({feature: None for feature in feature_fields})

        results.append(row)
    except requests.RequestException as exc:
        results.append(
            {
                "track_id": track_id,
                "status_code": None,
                "error_message": str(exc),
                **{feature: None for feature in feature_fields},
            }
        )

analysis_df = pd.DataFrame(results)
display(analysis_df)

,track_id,status_code,error_message,key,mode,tempo,camelot,duration,popularity,energy,danceability,happiness,acousticness,instrumentalness,liveness,speechiness,loudness
0,7s25THrKz86DM225dOYwnr,429,You have exceeded the DAILY quota for Requests...,None,None,None,None,None,None,None,None,None,None,None,None,None,None


In [69]:
merged_df = tracks_df.merge(analysis_df, on="track_id", how="left")

merged_df.head()

,track_id,track_name,artist,album,key,mode,tempo,camelot,duration,popularity,energy,danceability,happiness,acousticness,instrumentalness,liveness,speechiness,loudness
0,7s25THrKz86DM225dOYwnr,Respect,Aretha Franklin,I Never Loved a Man the Way I Love You,None,None,None,None,None,None,None,None,None,None,None,None,None,None
